# SMARD API output → downstream validation and forecast context

**Author:** Hari Raj Singh  
**Date:** 14 September 2026  
**Upstream notebook:** `notebooks/API-connection.ipynb`  
**Expected upstream output:** `data/smard.csv`

This notebook is deliberately **downstream** of the team's SMARD acquisition notebook. It does not replace or duplicate the acquisition step. The upstream notebook downloads six SMARD series — Wind Offshore, Wind Onshore, Solar, Grid Load, Residual Load, and Forecast Wind + Solar — and writes them to `data/smard.csv`.

The purpose here is to validate that output, make its time semantics visible, check residual-load consistency, and examine the forecast-like series as **context only**.

> **Information-set boundary:** a column named `Forecast Wind + Solar` is not automatically an admissible forecast-origin predictor or official benchmark. Publication timing, issue time, vintage/revision semantics, and the declared forecast origin still need explicit qualification.

## 1. Where the code is reused

The upstream `notebooks/API-connection.ipynb` already owns the SMARD API mechanics:

- base endpoint family;
- filter IDs;
- Germany region (`DE`);
- hourly/quarter-hour selection;
- weekly-package discovery;
- package download;
- exact-range trimming;
- CSV export.

This downstream notebook **does not copy the acquisition `fetch()` implementation by default**. It consumes the CSV produced upstream, so API logic stays in one place.

### How to add a new kind of SMARD data

1. Find the official SMARD filter ID in the SMARD API documentation.
2. Add it to `FILTERS` in `notebooks/API-connection.ipynb`.
3. State the unit, cadence and scientific meaning explicitly.
4. Rerun endpoint checks and acquisition there.
5. Regenerate `data/smard.csv`.
6. Rerun this downstream notebook.
7. Add source-specific analysis only after checking semantics.
8. If it is a forecast, prove publication/vintage timing before model admission.

That is the reuse path: **extend acquisition upstream; analyze downstream.**

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

EXPECTED_COLUMNS = [
    "Wind Offshore",
    "Wind Onshore",
    "Solar",
    "Grid Load",
    "Residual Load",
    "Forecast Wind + Solar",
]

def find_team_repo_root() -> Path:
    """Find a checkout containing notebooks/API-connection.ipynb and data/."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "notebooks" / "API-connection.ipynb").exists()
            and (candidate / "data").is_dir()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not find the team grid-stress repository root. "
        "Run this notebook from inside that checkout."
    )

ROOT = find_team_repo_root()
CSV = ROOT / "data" / "smard.csv"

if not CSV.exists():
    raise FileNotFoundError(
        f"{CSV} does not exist. Run notebooks/API-connection.ipynb first."
    )

print("Team repository root:", ROOT)
print("Upstream SMARD CSV:", CSV.relative_to(ROOT))


## 2. Load the upstream CSV exactly as exported

The upstream notebook exports a German-Excel-friendly CSV:

- separator `;`
- decimal mark `,`
- UTF-8 with BOM
- local Europe/Berlin timestamp text **without the UTC offset**

That last choice is convenient for Excel but important scientifically: once `+01:00` / `+02:00` is removed, the repeated autumn DST hour cannot be uniquely mapped back to a physical UTC instant from the CSV alone.

So this CSV is suitable for downstream inspection and local-clock EDA, but it is **not yet a canonical physical-time identity for leakage-sensitive modelling**.

In [ ]:
df = pd.read_csv(
    CSV,
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
)

if "timestamp" not in df.columns:
    raise KeyError("Expected a 'timestamp' column from the upstream API notebook.")

missing_expected = [c for c in EXPECTED_COLUMNS if c not in df.columns]
if missing_expected:
    raise KeyError(f"Missing expected SMARD columns: {missing_expected}")

df["timestamp_local_naive"] = pd.to_datetime(df["timestamp"], errors="raise")
df = df.sort_values("timestamp_local_naive", kind="stable").reset_index(drop=True)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("First local timestamp:", df["timestamp_local_naive"].min())
print("Last local timestamp:", df["timestamp_local_naive"].max())
display(df.head())


## 3. Quality audit: missingness, zeros and duplicated local clock times

`0` and missing are different states. Zero can be a valid physical observation, especially for solar or wind.

Because the upstream CSV strips the timezone offset, repeated autumn clock times can appear as duplicate local labels. We detect and report them rather than deleting or averaging them away.

In [ ]:
numeric_cols = [
    c for c in df.columns
    if c not in {"timestamp", "timestamp_local_naive"}
]

quality = pd.DataFrame(
    {
        "dtype": df[numeric_cols].dtypes.astype(str),
        "missing": df[numeric_cols].isna().sum(),
        "zeros": (df[numeric_cols] == 0).sum(),
        "min": df[numeric_cols].min(),
        "max": df[numeric_cols].max(),
    }
)

duplicate_mask = df["timestamp_local_naive"].duplicated(keep=False)
duplicate_rows = int(duplicate_mask.sum())
duplicate_labels = int(
    df.loc[duplicate_mask, "timestamp_local_naive"].nunique()
)

display(quality)
print("Duplicate local-clock labels retained in exported CSV:", duplicate_rows)
print("Distinct duplicated local-clock labels retained in exported CSV:", duplicate_labels)
print("Total missing cells:", int(df[numeric_cols].isna().sum().sum()))

if duplicate_rows:
    print(
        "\nDo not deduplicate these blindly: repeated local labels may represent "
        "different physical hours around the autumn DST transition."
    )
    display(df.loc[duplicate_mask, ["timestamp", *EXPECTED_COLUMNS]].head(12))


## Time integrity gate — DST fall-back loss in the upstream CSV

The exported `timestamp` column contains timezone-naive Europe/Berlin clock labels.
A zero duplicate count is therefore **not evidence of lossless time coverage**.

For a timezone-aware hourly series, a spring DST transition contains 23 physical
hours and an autumn transition contains 25. We compare the exported CSV against
the expected Europe/Berlin calendar below.

If an autumn day contains only 24 rows, one of the two physical observations
sharing the repeated local `02:00` label has been lost upstream.

**Consequence:** this CSV is suitable for descriptive downstream work only with
this limitation made explicit. The naive timestamp export must not be treated as
a canonical physical-time index. A future upstream fix should preserve the
original epoch/UTC timestamp as the unique key and derive local Berlin time as an
additional analysis/display column.


In [ ]:
DECLARED_START_LOCAL = pd.Timestamp("2022-01-01 00:00", tz="Europe/Berlin")
DECLARED_END_LOCAL = pd.Timestamp("2026-09-08 00:00", tz="Europe/Berlin")

expected_index = pd.date_range(
    start=DECLARED_START_LOCAL,
    end=DECLARED_END_LOCAL,
    freq="h",
    inclusive="left",
)

expected_day_counts = (
    pd.Series(1, index=expected_index)
    .groupby(expected_index.date)
    .sum()
)

observed_day_counts = (
    df.groupby(df["timestamp_local_naive"].dt.date)
    .size()
)

time_audit = pd.DataFrame(
    {
        "expected_physical_hours": expected_day_counts,
        "exported_rows": observed_day_counts,
    }
).fillna(0).astype(int)

time_audit["difference"] = (
    time_audit["exported_rows"] - time_audit["expected_physical_hours"]
)

dst_or_mismatch = time_audit[
    (time_audit["expected_physical_hours"] != 24)
    | (time_audit["exported_rows"] != time_audit["expected_physical_hours"])
]

missing_physical_observations = int(
    (
        time_audit["expected_physical_hours"]
        - time_audit["exported_rows"]
    )
    .clip(lower=0)
    .sum()
)

mismatch_days = time_audit[time_audit["difference"] != 0]

print("Expected physical hourly observations:", len(expected_index))
print("Exported CSV rows:", len(df))
print("Missing physical observations:", missing_physical_observations)
print()
print("DST / coverage audit:")
display(dst_or_mismatch)

print()
if mismatch_days.empty:
    print("TIME INTEGRITY GATE: PASS")
else:
    print("TIME INTEGRITY GATE: FAIL")
    print(
        "The exported naive-local timestamp series has lost "
        f"{missing_physical_observations} physical observation(s)."
    )
    print("Mismatching calendar days:")
    display(mismatch_days)


## 4. Residual-load consistency check

For these selected series:

`derived residual load = Grid Load - Wind Offshore - Wind Onshore - Solar`

We compare that derived quantity with SMARD's supplied `Residual Load`.

This is a consistency check, not an attempt to redefine SMARD. If the two differ materially, the difference needs investigation before either is used analytically.

In [ ]:
df["Realized Wind + Solar"] = (
    df["Wind Offshore"] + df["Wind Onshore"] + df["Solar"]
)
df["Derived Residual Load"] = (
    df["Grid Load"] - df["Realized Wind + Solar"]
)
df["Residual Load Difference"] = (
    df["Residual Load"] - df["Derived Residual Load"]
)

residual_check = pd.Series(
    {
        "mean_abs_difference_mw": df["Residual Load Difference"].abs().mean(),
        "max_abs_difference_mw": df["Residual Load Difference"].abs().max(),
        "correlation": df[
            ["Residual Load", "Derived Residual Load"]
        ].corr().iloc[0, 1],
    }
)

display(residual_check.to_frame("value"))


## 5. Forecast-vs-realized diagnostic — descriptive only

The upstream dataset contains `Forecast Wind + Solar`. We compare it with realized wind + solar to understand historical forecast behaviour.

This **does not yet establish an admitted benchmark**. The CSV does not encode issue time or forecast vintage. A historical forecast value is usable at a declared forecast origin only if we can prove which version was available at that time.

In [ ]:
df["Wind + Solar Forecast Error"] = (
    df["Forecast Wind + Solar"] - df["Realized Wind + Solar"]
)

forecast_error = df["Wind + Solar Forecast Error"].dropna()

forecast_metrics = pd.Series(
    {
        "observations": len(forecast_error),
        "mean_error_mw": forecast_error.mean(),
        "mae_mw": forecast_error.abs().mean(),
        "rmse_mw": np.sqrt(np.mean(forecast_error**2)),
        "p05_error_mw": forecast_error.quantile(0.05),
        "median_error_mw": forecast_error.median(),
        "p95_error_mw": forecast_error.quantile(0.95),
    }
)

display(forecast_metrics.to_frame("value"))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(forecast_error, bins=80)
ax.axvline(0, linewidth=1)
ax.set(
    title="SMARD — Forecast Wind + Solar minus realized Wind + Solar",
    xlabel="Forecast error (MW)",
    ylabel="Hourly observations",
)
ax.grid(alpha=0.2)
plt.show()


## 6. Concise local-time EDA

The local timestamps remain useful for descriptive temporal structure. We do **not** pretend the naive CSV timestamps are a canonical UTC index.

In [ ]:
df["local_hour"] = df["timestamp_local_naive"].dt.hour
df["month"] = df["timestamp_local_naive"].dt.month

hour_profile = (
    df.groupby("local_hour")[
        ["Grid Load", "Residual Load", "Realized Wind + Solar"]
    ]
    .mean()
)

fig, ax = plt.subplots(figsize=(9, 4.8))
for column in hour_profile.columns:
    ax.plot(
        hour_profile.index,
        hour_profile[column],
        marker="o",
        label=column,
    )

ax.set(
    title="SMARD — average local-hour power profile",
    xlabel="Local hour (Europe/Berlin clock label)",
    ylabel="Mean power (MW)",
    xticks=range(0, 24, 2),
)
ax.legend()
ax.grid(alpha=0.25)
plt.show()

month_profile = (
    df.groupby("month")[
        [
            "Grid Load",
            "Residual Load",
            "Wind Onshore",
            "Wind Offshore",
            "Solar",
        ]
    ]
    .mean()
)

display(month_profile)


## 7. Modelling relevance

This SMARD output is closer to the eventual GridStress forecasting problem than a purely contextual source because it contains both realized system quantities and a forecast-like series.

Potential later roles:

- **targets/outcomes:** realized Grid Load or Residual Load;
- **lagged predictors:** past load/residual load, if lagged strictly before the forecast origin;
- **regime/context variables:** calendar structure, historical wind/solar, ramps and mix;
- **official operational benchmark candidate:** `Forecast Wind + Solar`, but only after publication/vintage timing is proven.

The current CSV proves that historical values exist. It does **not** prove what was knowable in real time.

## 8. Exact place to repurpose the API code for new series

The upstream notebook's extension point is the `FILTERS` mapping.

Current upstream examples:

- `1225` → Wind Offshore
- `4067` → Wind Onshore
- `4068` → Solar
- `410` → Grid Load
- `4359` → Residual Load
- `5097` → Forecast Wind + Solar

For another SMARD series:

1. edit `FILTERS` in `notebooks/API-connection.ipynb`;
2. add the official filter ID and clear name;
3. reuse the existing weekly-package `fetch()` logic there;
4. rerun endpoint checks and download;
5. regenerate `data/smard.csv`;
6. rerun this notebook;
7. add a dedicated check only if the new series has a justified scientific role;
8. document any different unit, cadence, geography, sign convention or publication timing explicitly.

So we are reusing the upstream code **where it belongs**, while keeping downstream analysis separate.

## 9. What we validated / what still needs qualification

### What this notebook can establish

- the upstream CSV exists and is readable;
- expected SMARD columns are present;
- missing values, zeros and duplicate local-clock labels are explicit;
- residual-load arithmetic can be checked;
- forecast-vs-realized behaviour can be examined descriptively;
- intraday and seasonal structure can be explored without building a model.

### What still needs qualification

- canonical physical-time identity should retain UTC or offsets end-to-end;
- repeated autumn local hours must not be collapsed;
- exact forecast issue/publication time;
- forecast vintage and revision semantics;
- reconstruction of what was known at the declared forecast origin;
- final assignment of each series as descriptive context, target/outcome, benchmark or predictor.

Until those timing questions are resolved, this remains a **downstream validation and modelling-readiness notebook**, not a claim that the final information set is frozen.